In [9]:
import numpy as np
import pandas as pd
import joblib
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

from lightgbm import LGBMClassifier
from sklearn.calibration import CalibratedClassifierCV

import shap

In [10]:
from google.colab import files

uploaded = files.upload()

Saving severity_dataset_dropped_correlated_columns.csv to severity_dataset_dropped_correlated_columns (1).csv


In [11]:
df = pd.read_csv(
    "severity_dataset_dropped_correlated_columns.csv"
)

print(df.shape)

df.head()

(489, 70)


,Unnamed: 0,wrist_mvmnt_x_median,wrist_mvmnt_x_min,wrist_mvmnt_y_median,wrist_mvmnt_y_min,wrist_mvmnt_y_max,wrist_mvmnt_dist_min,aperiodicity_denoised,aperiodicity_trimmed,periodEntropy_denoised,...,Comment1,Comment3,Comment4,CheckDifficult1,CheckDifficult3,CheckDifficult4,gender,age,diagnosed,Rating
0,0,4.238080,0.000000,8.462262,0.000000,35.881219,0.000000,1.408849,1.245572,0.320929,...,Unnamed: 248,Unnamed: 248,NaN,False,False,False,female,78.0,yes,2
1,1,2.353139,0.000000,5.643571,0.000000,27.636004,0.000000,2.073631,1.723751,0.487256,...,Unnamed: 158,Unnamed: 158,NaN,False,False,False,male,69.0,yes,2
2,2,2.414284,0.110869,12.464668,0.000000,69.834736,3.234005,2.206180,1.678927,0.325414,...,Unnamed: 152,Unnamed: 152,NaN,False,False,False,male,72.0,yes,1
3,3,22.882927,0.000000,28.877301,0.000000,207.739263,0.000000,1.157455,1.061825,0.452998,...,Unnamed: 34,Unnamed: 34,NaN,False,False,False,male,86.0,yes,3
4,4,1.203146,0.040405,6.464794,0.129821,45.059041,0.698013,1.986317,1.278567,0.256317,...,Unnamed: 78,Unnamed: 78,NaN,False,False,False,male,36.0,no,0


In [12]:
drop_columns = [
    "Unnamed: 0",
    "filename",
    "Comment1",
    "Comment3",
    "Comment4",
    "Rating",
    "Rating1",
    "Rating2",
    "Rating3",
    "Rating4",
    "CheckDifficult1",
    "CheckDifficult3",
    "CheckDifficult4",
]

df = df.drop(columns=drop_columns)

print(df.shape)

(489, 57)


In [13]:
label_hand = LabelEncoder()
label_gender = LabelEncoder()

df["hand"] = label_hand.fit_transform(df["hand"])

df["gender"] = label_gender.fit_transform(df["gender"])

df["diagnosed"] = (
    df["diagnosed"]
    .map({
        "no": 0,
        "yes": 1
    })
    .astype(int)
)

In [14]:
X = df.drop(columns=["diagnosed"])
y = df["diagnosed"]

print(X.shape)
print(y.shape)
print(y.value_counts())

(489, 56)
(489,)
diagnosed
1    333
0    156
Name: count, dtype: int64


In [15]:
from sklearn.model_selection import train_test_split

# 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(342, 56)
(73, 56)
(74, 56)


In [16]:
from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_val = scaler.transform(X_val)

X_test = scaler.transform(X_test)

joblib.dump(
    scaler,
    "scaler.pkl"
)

['scaler.pkl']

In [17]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    random_state=42
)

lgbm.fit(
    X_train,
    y_train
)

[LightGBM] [Info] Number of positive: 233, number of negative: 109
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000578 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4998
[LightGBM] [Info] Number of data points in the train set: 342, number of used features: 56
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.681287 -> initscore=0.759691
[LightGBM] [Info] Start training from score 0.759691
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

LGBMClassifier(learning_rate=0.03, n_estimators=500, random_state=42)

In [18]:
from sklearn.calibration import CalibratedClassifierCV

calibrated_lgbm = CalibratedClassifierCV(
    lgbm,
    method="isotonic",
    cv="prefit"
)

calibrated_lgbm.fit(
    X_val,
    y_val
)

/usr/local/lib/python3.12/dist-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


CalibratedClassifierCV(cv='prefit',
                       estimator=LGBMClassifier(learning_rate=0.03,
                                                n_estimators=500,
                                                random_state=42),
                       method='isotonic')

In [19]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

preds = calibrated_lgbm.predict(X_test)

probs = calibrated_lgbm.predict_proba(X_test)[:, 1]

print(confusion_matrix(y_test, preds))

print(classification_report(y_test, preds))

print("ROC AUC:",
      roc_auc_score(y_test, probs))

[[10 14]
 [ 0 50]]
              precision    recall  f1-score   support

           0       1.00      0.42      0.59        24
           1       0.78      1.00      0.88        50

    accuracy                           0.81        74
   macro avg       0.89      0.71      0.73        74
weighted avg       0.85      0.81      0.78        74

ROC AUC: 0.8404166666666667


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [20]:
import shap
import numpy as np

background = X_train[
    np.random.choice(
        len(X_train),
        size=min(100, len(X_train)),
        replace=False
    )
]

explainer = shap.TreeExplainer(
    calibrated_lgbm.estimator
)

shap_values = explainer.shap_values(background)

print("SHAP explainer created successfully.")

SHAP explainer created successfully.


/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [21]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": calibrated_lgbm.estimator.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance.head(20)

,Feature,Importance
55,age,605
37,amplitude_decrement_end_to_mean_denoised,313
23,frequency_lr_fitness_r2_denoised,210
4,wrist_mvmnt_y_max,202
50,speed_min_trimmed,185
51,acceleration_min_denoised,172
17,period_quartile_range_denoised,167
0,wrist_mvmnt_x_median,156
33,amplitude_entropy_denoised,148
5,wrist_mvmnt_dist_min,146


In [22]:
joblib.dump(
    list(X.columns),
    "selected_feature_names.pkl"
)

print("Feature list saved.")

Feature list saved.


In [23]:
joblib.dump(
    calibrated_lgbm,
    "finger_tapping_model.pkl"
)

print("Model exported.")

Model exported.


In [24]:
auc = roc_auc_score(
    y_test,
    calibrated_lgbm.predict_proba(X_test)[:,1]
)

metadata = {
    "model_id": "finger_tapping_lgbm_v1",
    "validation_auc": float(auc),
    "num_features": int(X.shape[1]),
    "dataset_size": int(len(df)),
    "target": "diagnosed"
}

with open(
    "metadata.json",
    "w"
) as f:
    json.dump(
        metadata,
        f,
        indent=2
    )

print(metadata)

{'model_id': 'finger_tapping_lgbm_v1', 'validation_auc': 0.8404166666666667, 'num_features': 56, 'dataset_size': 489, 'target': 'diagnosed'}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [25]:
import os

for f in [
    "finger_tapping_model.pkl",
    "scaler.pkl",
    "selected_feature_names.pkl",
    "metadata.json"
]:
    print(f, os.path.exists(f))

finger_tapping_model.pkl True
scaler.pkl True
selected_feature_names.pkl True
metadata.json True


In [26]:
from google.colab import files

files.download("finger_tapping_model.pkl")
files.download("scaler.pkl")
files.download("selected_feature_names.pkl")
files.download("metadata.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>